# 00_00 Course map: build Greek LLMs from Homer

*Our own page: not in the book.*

Start here. This page covers the questions you'll probably have on day one.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **the start**, looking at the whole map

## 1. What you'll learn

- What we're building, and where it ends up
- The path the Greek text takes through the project
- Which chapter teaches which part, and in what order
- How to use a notebook in VS Code

## 2. Words you'll meet

The new words, explained before we use them. Come back to this table whenever one slips your mind.

| Word | Meaning in plain English | Example |
|---|---|---|
| **LLM** | large language model: a program that reads text and guesses what comes next | after `ἄνδρα μοι` it guesses `ἔννεπε` |
| **notebook** | a file of small boxes called cells; each cell holds code or notes | this file, `00_00_course_map.ipynb` |
| **cell** | one box in a notebook; you run it and its result appears underneath | the box you're reading now |
| **kernel** | the Python program that runs your notebook's cells in the background | the one in `.venv` |
| **MCP** | Model Context Protocol: the standard way for AI apps like Claude to use outside tools | our `homer_mcp_server.py` |

## 3. The idea

### ❓ What are we actually building?

Have you ever sung along to a song in a language you don't speak? After enough listens you know which sound comes next, and you still have no idea what the words mean. That's what we build here, for Homer's Greek: a program that goes through the poem over and over, practises guessing what comes next, and ends up writing lines of its own that sound like Homer. At the very end, other AI apps like Claude can ask it for a line, the way a friend might ask you to hum that song. The picture has one limit: you started with ears and a brain, and our program starts as a pile of random numbers, so most of the course goes into building those.

The proper name for such a program is a **large language model**, or **LLM** for short: a program that reads some text and guesses what comes next. Ours learns from Homer's *Odyssey*, in the original Greek, so it learns to write Greek that sounds like Homer.

Really, the whole course is about making those guesses better. On the way, the Greek text passes through six stages:

```
Odyssey text → numbers → model reads → model learns → model writes → MCP tool
```

The last stage, the MCP tool, is how other AI apps, like Claude, can ask our model for Greek text.

### ❓ What happens to one word on the way through?

Let's follow `ἄνδρα` ("man", the Odyssey's first word) through our real, trained model. The **tokenizer** gives each character a number. The **model** turns those numbers into a guess for the next character. Then we add the best guess to the end and guess again.

And these are real numbers, from the small model in this repo. You'll meet that model in [01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb), and print these very chances yourself in [06_01](../06_making_it_talk/06_01_writing_greek.ipynb).

**Look at:** the four rows, top to bottom. The orange bar is the model's best guess, and the green square is the letter that gets added.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    EDGE,
    GOOD,
    INK,
    MUTED,
    arrow,
    chips,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("One word's journey", "What happens to ἄνδρα", (9, 7.5))
ax = diagram_axes(fig)


def row(y, labels, colors=None):
    left = 4.0 - (len(labels) * 0.9 - 0.1) / 2  # centre the row on x = 4
    chips(ax, left, y, labels, 0.8, 0.1, colors)


row(6.7, list("ἄνδρα"))
arrow(ax, (4, 6.15), (4, 5.65), "tokenizer (chapter 2)", label_offset=(1.15, 0))
row(5.2, ["58", "38", "29", "42", "26"])
arrow(
    ax,
    (4, 4.65),
    (4, 4.3),
    "model (chapters 3-4), trained in chapter 5",
    label_offset=(2.2, 0),
)

for i, (letter, chance) in enumerate([("␣", 0.26), ("ι", 0.15), ("τ", 0.07)]):
    bar_color = ACCENT if i == 0 else MUTED
    ax.bar(3 + i, chance * 3.5, width=0.6, bottom=2.95, color=bar_color)
    tag(ax, 3 + i, 3.12 + chance * 3.5, f"{chance:.2f}", INK, 9, mono=True)
    tag(ax, 3 + i, 2.7, letter, INK, 12, mono=True)

arrow(
    ax,
    (4, 2.35),
    (4, 1.75),
    "add the best guess, then guess again (chapter 6)",
    label_offset=(2.6, 0),
)
row(1.25, list("ἄνδρα␣"), [EDGE] * 5 + [GOOD])
tag(ax, 4, 0.5, "and so on: ἄνδρα τοι και …", INK, 10)

for y, text in [
    (6.7, "1. the text"),
    (5.2, "2. numbers"),
    (3.4, "3. the model's chances"),
    (1.25, "4. written text"),
]:
    tag(ax, 9.6, y, text, MUTED, 9, ha="right")
plt.show()

### ❓ Where are we, and what do we have so far?

Right at the start, with nothing built yet. You don't need to know anything about AI. Every new AI word gets explained before we use it, and every idea comes with a tiny example you can try.

### ❓ What problem does this page solve?

A course with this many notebooks is easy to get lost in. So this page is the map. It shows every chapter, what it teaches and where it fits, so you always know where you are, and why you're doing what you're doing.

### The guessing machine, and how it uses tools

### ❓ If an LLM only guesses the next token, how can Claude "use a tool" or "take an action"?

It can't, not by itself. The model never runs anything; it only writes text, one **token** at a time (a token is one piece of text; for our model, one character). The trick is in the training: when a tool would help, the most likely next tokens are a special, strictly formatted message. Think of it as filling in a form.

### ❓ What does that "form" look like?

Something like `{"tool": "consult_homer_llm", "greek_seed": "ἄνδρα"}`. That way of writing data is called **JSON**: plain text, with names and values in curly brackets. It's often wrapped in special tokens that mean "a tool call starts here" and "it ends here". Same idea as our `<|endoftext|>` marker in [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb).

### ❓ Then who actually runs the tool?

The program around the model. People often call it the **harness**: the app that feeds text in and reads text out. It watches the tokens as they come out, and when it spots a tool-call form, it stops the model, runs the tool (say, our MCP server's `consult_homer_llm`) and pastes the result back into the conversation as more text.

### ❓ And then?

The model just carries on guessing next tokens. Only now the tool's answer is right there in front of it, so its next words can use it:

```
guess → guess → [tool form] → app runs the tool → [result text] → guess → guess
```

### ❓ How did the model learn to write the form at the right moment?

Extra training, on lots of example conversations where the right move was a tool call. Extra training like this, on top of a finished model, is called **fine-tuning** (chapter 8 does a small one, on GPT-2). That's way beyond our toy model, though, which only writes Greek. So in this repo it's Claude that writes the tool-call form, and **our model is the tool**.

### ❓ How does Claude know our tool exists, and when to call it?

Have a look in `src/homer_ai/homer_mcp_server.py`. The line `@mcp.tool()` registers the function `consult_homer_llm` as a tool. Its **docstring** (the text in triple quotes under the function) is the description Claude reads when it decides whether to call it.

### ❓ What is the end goal?

Three models, one after the other:

```
 Odyssey model  →  Iliad model  →  combine both
 (chapters 1-6)    (same steps)    (chapter 7)
```

1. A model that learns from the **Odyssey**.
2. The same steps again for the **Iliad**, Homer's other great poem.
3. In advanced chapter 7, the two **combined**.

After that, advanced chapter 8 borrows a model somebody else trained (GPT-2) and fine-tunes it for a new job, and advanced chapter 9 answers questions about Homer from the real text, with the book and line beside each answer. Advanced chapter 10 leaves Homer and does the same three things (look up, fine-tune, measure) on the kind of text a company has: product sheets, customer emails and service-desk tickets, all invented.

### ❓ Which chapters are there?

| Chapter | What you learn | Stage |
|---|---|---|
| `00_start_here` | The course map, setup, and reading error messages | getting ready |
| `01_foundations` | What a language model is; a model made of counting; neural networks; how a program learns; our model as a black box | the basics |
| `02_text_to_numbers` | Turning Greek text into numbers the model can use | numbers |
| `03_attention` | How the model decides which earlier letters matter | model reads |
| `04_building_the_model` | Putting the parts together into a full model | model reads |
| `05_training_on_the_odyssey` | Teaching the model, and checking it really learned | model learns |
| `06_making_it_talk` | Writing Greek, and offering it to other AI apps | model writes, MCP tool |
| `07_advanced_iliad_and_combining` | The Iliad model, comparing models fairly, and combining both | all of them again |
| `08_advanced_fine_tuning` | Fine-tuning a pretrained model: GPT-2's weights in our own class, a spam filter, instructions, LoRA | a borrowed model |
| `09_advanced_grounded_answers` | Answers grounded in the real text: passages, a word search (BM25) from scratch, scoring the search, a second MCP tool | Odyssey text, MCP tool |
| `10_advanced_your_domain` | The same pattern on a company's own text, with invented data: support answers with citations, spotting complaints in email, service-desk triage, and the pitfalls | your own data, MCP tool |
| `A_toolbox` | Appendix: PyTorch, the dot product, tiktoken, and the other tools; dip in any time | all of them |

Every chapter folder also has a `README.md` that lists its notebooks in order.

### ❓ What's the toolbox, and when do I use it?

[Appendix A, the toolbox](../A_toolbox/A_00_overview.ipynb), covers the tools we use: PyTorch, the dot product, tiktoken, uv, Ruff, Jupyter, git and MCP. It's not something you read first. You dip into it whenever a tool comes up that you don't know yet. New to tensors, say? That's A_02.

### ❓ Do I need the book?

No. The course follows Sebastian Raschka's book *Build a Large Language Model (From Scratch)*, but every idea is explained in full here in the notebooks. If you do have the book, each notebook tells you which section it matches. And where a beginner needs more background than the book gives, we've added chapters of our own.

### ❓ What does every notebook look like?

The same parts, in the same order, every time:

1. What you'll learn
2. Words you'll meet
3. The idea: background first, then the idea, as questions and answers
4. Step by step: one question per step; predict, run, look
5. A full example on the real Greek text (when the topic has one)
6. The questions other learners asked, then a recap

(The section numbers just count up, so a notebook with no full example has one number fewer.)

## 4. Step by step: how to start

### ❓ How do I get set up?

Go through [00_01: Setup](00_01_setup.ipynb). You install VS Code, git and uv, grab the project, run `uv sync` and pick the `.venv` kernel. The last cell there checks that it all works.

### ❓ What if I get an error?

You will, everyone does. [00_02: Reading errors](00_02_reading_errors.ipynb) shows you how to read one, using the actual errors we hit while building this project.

### ❓ Where do I start learning?

With Chapter 1, [01_00: Foundations](../01_foundations/01_00_chapter_overview.ipynb). Then just go through the chapters in order. If you meet a tool or a word you don't know (**tensor** is a likely one), dip into [Appendix A, the toolbox](../A_toolbox/A_00_overview.ipynb). For tensors, that's [A_02](../A_toolbox/A_02_pytorch_tensors.ipynb).

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Who is this course for?

Developers who already know some language well (it doesn't have to be Python) but have never done machine learning. School maths is enough. Matrices, `e` and logarithms get a gentle introduction when they first show up. We explain the AI and the PyTorch, not general programming. (New to Python? [A_01](../A_toolbox/A_01_python_essentials.ipynb) is an optional primer.)

*I came to this from years of C#, .NET and TypeScript, not Python. So I explain the AI and the PyTorch, not the language.*

### ❓ Can I break anything by running the notebooks?

No, you can't. The notebooks only read the Greek text and run short demos. They never touch the project's code or the trained models.

## 5. Recap

- We're building LLMs that write Homer's Greek: the Odyssey first, then the Iliad, then both together.
- The text's path: text → numbers → model reads → model learns → model writes → MCP tool.
- Set up with [00_01](00_01_setup.ipynb), then start on Chapter 1. The toolbox is there whenever you need it.

**Next:** [00_01: Setup](00_01_setup.ipynb).